# 3D Neuron Visualizer

Generate interactive Three.js visualizations from neuPrint connectomics data.

---

### neuPrint Credentials

**NEUPRINT_TOKEN** — Your neuPrint API token.  
Get yours at [neuprint-cns.janelia.org](https://neuprint-cns.janelia.org/) → Account → Auth Token.

**NEUPRINT_SERVER** — neuPrint server URL (default: `neuprint-cns.janelia.org`)  
**NEUPRINT_DATASET** — Dataset to query (default: `cns`)

---

### Inputs

**PATTERN** — neuPrint regex for neuron types  
Examples: `'^FB.*'`, `'^EPG.*'`, `'^LNO|GLNO|LCNO.*'`

---

### Color Modes

In addition to the four built-in modes (*Cell Type*, *Instance*, *Predicted NT*, *Custom*),
you can add your own by providing CSVs. There are two types:

#### Continuous (`CONTINUOUS_CSVS`)

Each CSV must have a `type` column whose values match neuPrint type names.  
Every other column becomes a separate continuous color mode, where each type is colored
by its numeric value along a colormap.

- **Divergent data** (has both positive and negative values) → auto-assigned `RdBu` (red–white–blue) colormap  
- **Non-divergent data** (all values ≥ 0) → auto-assigned a sequential colormap (Oranges, Purples, Greens, …)

You can supply **multiple CSVs** in the list — columns across all of them become separate color modes.

```
Example CSV (single score per type, divergent):
  type,valence_score
  FB2M_a,0.093
  FB5J,-0.041

Example CSV (multiple scores per type, non-divergent):
  type,olfactory_input,gustatory_input,visual_input
  FB1A,0.0034,0.0052,0.00002
  FB1B,0.0072,0.0033,0.00014
```

#### Categorical (`CATEGORICAL_CSVS`)

Each CSV must have a `type` column whose values match neuPrint type names.  
Every other column becomes a categorical color mode, where types sharing the same
category value are assigned the same color.

```
Example CSV:
  type,layer,transmitter_class
  FB1A,dorsal,inhibitory
  FB1B,ventral,excitatory
  FB2C,dorsal,inhibitory
```

In [ ]:
import sys
sys.path.insert(0, 'Core_Code')
from generate_visualization import generate_visualization

# --- neuPrint credentials ---

NEUPRINT_TOKEN = 'YOUR_TOKEN_HERE'
# Paste your token from neuprint-cns.janelia.org → Account → Auth Token

NEUPRINT_SERVER = 'neuprint-cns.janelia.org'

NEUPRINT_DATASET = 'cns'

# --- Neuron pattern ---

PATTERN = '^FB.*'
# neuPrint regex for neuron types. Examples: '^EPG.*', '^LNO|GLNO|LCNO.*'

# --- Color modes (set to [] or None to skip) ---

CONTINUOUS_CSVS = [
    'Color_by_CSVs/fb_type_scores.csv',
    'Color_by_CSVs/fbt_sensory_modality_scores_unsigned.csv',
]
# Each CSV needs a 'type' column. Every other column becomes a continuous color mode.

CATEGORICAL_CSVS = [
    # 'Color_by_CSVs/my_categories.csv',
]
# Each CSV needs a 'type' column. Every other column becomes a categorical color mode.

# --- Neuron meshes ---

USE_MESHES = True
# Fetch and render 3D neuron surfaces. Skeletons are always included regardless.

MESH_FACES = 'auto'
# Controls mesh decimation (simplification). Options:
#   'auto' — Maximize mesh quality while keeping the HTML under MAX_FILE_MB.
#             Automatically computes the best face count per neuron.
#   50000  — Fixed face count per neuron. Good for ~50 cells.
#   20000  — Lower detail, smaller files. Good for 500+ cells.
#   None   — No decimation. Full resolution (~130K-190K faces/neuron). Very large files.
# Note: fetching meshes from neuPrint is quick — decimation is the slow part.
# But decimation time is roughly the same regardless of target face count, so
# set this based on desired file size and quality, not generation speed.

MAX_FILE_MB = 500
# Target file size limit in MB. Only used when MESH_FACES = 'auto'.
# Larger = higher mesh quality. Smaller = faster loading in the browser.

# --- Synapse data ---

LOAD_SYNAPSES = True
# Fetch synapse positions and embed them in the HTML.
# This is the SLOWEST step — it can take 5-20+ minutes depending on neuron count.
# The first time you run this, positions are fetched from neuPrint and saved as a
# .json file right next to your HTML (e.g. Non_Committal/FB_synapses.json).
# On subsequent runs, that .json is reused automatically so you don't wait again.
# Keep that .json around! Deleting it means re-fetching from scratch.

RELOAD_SYNAPSES = False
# Set True ONLY if you've changed your search term (PATTERN) or dataset and need
# fresh synapse data. This deletes the existing .json and re-fetches everything.

# --- Generate HTML ---

generate_visualization(
    PATTERN,
    continuous_csvs=CONTINUOUS_CSVS or None,
    categorical_csvs=CATEGORICAL_CSVS or None,
    use_meshes=USE_MESHES,
    mesh_faces=MESH_FACES,
    max_file_mb=MAX_FILE_MB,
    server=NEUPRINT_SERVER,
    dataset=NEUPRINT_DATASET,
    token=NEUPRINT_TOKEN,
    skip_synapses=True,
    auto_open=True,
)

In [ ]:
import re
from pathlib import Path
from add_synapses import add_synapses_to_html
from generate_visualization import get_client

if LOAD_SYNAPSES:
    get_client(server=NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=NEUPRINT_TOKEN)

    safe_name = re.sub(r'[\^$.*+?\[\](){}|\\]', '', PATTERN)
    html_path = Path('Non_Committal') / f'{safe_name}_visualization.html'
    cache_path = html_path.parent / f'{safe_name}_synapses.json'

    if RELOAD_SYNAPSES and cache_path.exists():
        cache_path.unlink()
        print(f'Deleted {cache_path.name} — will re-fetch from neuPrint')

    add_synapses_to_html(html_path)
else:
    print('Skipping synapse fetch (LOAD_SYNAPSES = False)')